# Distribution comparison: case counts per grid cell

Counts cases per population-grid cell for the original dataset and every masked variant and reports summary statistics (mean, variance, quantiles, share of cases lost relative to the original), as a first, purely quantitative utility check before the spatial analyses.

**Input:** the datasets listed in `DATASETS` below, plus the population grid.
**Output:** `outputs_distribution_comparison/distribution_comparison_case_counts.csv`.


In [ ]:
# Imports
import os
from pathlib import Path
import pandas as pd
import geopandas as gpd


In [ ]:
# Paths
# Data directory. Override with the DATA_DIR environment variable to
# point at a different copy of the data without editing this file, e.g.
#   DATA_DIR=/path/to/data jupyter execute this_notebook.ipynb
# The data itself is not part of this repository -- see README.md.
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))

GRID_PATH = DATA_DIR / "population_grid" / "GHS_Hungary_pop_city-clipped2.shp"

DATASETS = {
    "original": DATA_DIR / "original" / "2008_eov_illness_stand.shp",
    "donut": DATA_DIR / "donut_masking" / "donut_points150_high900.shp",
    "statmasked": DATA_DIR / "statistical_masking" / "statmasked_points.shp",
    "synthetic": DATA_DIR / "synthetic" / "synthetic_points.shp",
}

OUT_DIR = Path("outputs_distribution_comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUT_DIR / "distribution_comparison_case_counts.csv"

In [ ]:
# Load grid and prepare for spatial join
grid = gpd.read_file(GRID_PATH).reset_index(drop=True)
grid["grid_id"] = grid.index

rows = []
original_total = None

# For each dataset, count points in each grid cell and calculate statistics
for name, path in DATASETS.items():
    points = gpd.read_file(path)

    # Use the same CRS for points and grid
    if points.crs != grid.crs:
        points = points.to_crs(grid.crs)

    # Find which points fall within which grid cells
    joined = gpd.sjoin(
        points[["geometry"]],
        grid[["grid_id", "geometry"]],
        how="inner",
        predicate="within"
    )

    # Include all grid cells, including cells with zero points
    counts = joined.groupby("grid_id").size().reindex(grid["grid_id"], fill_value=0)

    total_cases = counts.sum()
    if name == "original":
        original_total = total_cases

    rows.append({
        "dataset": name,
        "n_grid_cells": len(grid),
        "n_cases_in_grid": int(total_cases),
        "points_outside_grid": int(len(points) - len(joined)),
        "n_missing": None if original_total is None else round((original_total - total_cases) / original_total * 100, 2),
        "mean": counts.mean(),
        "median": counts.median(),
        "variance": counts.var(),
        "min": counts.min(),
        "p05": counts.quantile(0.05),
        "p25": counts.quantile(0.25),
        "p75": counts.quantile(0.75),
        "p95": counts.quantile(0.95),
        "max": counts.max(),
    })

result = pd.DataFrame(rows)
result.to_csv(OUTPUT_CSV, index=False)
result